# CSCI 3202
# Final Project
# Carter Sammis

In [1]:
cd aima-python/

/home/jovyan/CSCI 3202/Final_Project/aima-python


### Import Statements:

In [2]:
import random
import time

import sys
import io
from contextlib import contextmanager

### Class Definitions:

In [3]:
class Mancala:
    def __init__(self, pits_per_player, stones_per_pit):
        """
        The constructor for the Mancala class defines several instance variables:

        pits_per_player: This variable stores the number of pits each player has.
        stones_per_pit: It represents the number of stones each pit contains at the start of any game.
        board: This data structure is responsible for managing the Mancala board.
        current_player: This variable takes the value 1 or 2, as it's a two-player game, indicating which player's turn it is.
        moves: This is a list used to store the moves made by each player. It's structured in the format (current_player, chosen_pit).
        p1_pits_index: A list containing two elements representing the start and end indices of player 1's pits in the board data structure.
        p2_pits_index: Similar to p1_pits_index, it contains the start and end indices for player 2's pits on the board.
        p1_mancala_index and p2_mancala_index: These variables hold the indices of the Mancala pits on the board for players 1 and 2, respectively.
        """
        self.pits_per_player = pits_per_player
        self.board = [stones_per_pit] * ((pits_per_player+1) * 2)  # Initialize each pit with stones_per_pit number of stones 
        self.players = 2
        self.current_player = 1
        self.moves = []
        self.p1_pits_index = [0, self.pits_per_player-1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player+1, len(self.board)-1-1]
        self.p2_mancala_index = len(self.board)-1
        
        # Zeroing the Mancala for both players
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        """
        Displays the board in a user-friendly format
        """
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1]+1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1]+1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            if i == self.pits_per_player - 1:
                print('{} -> |_{}_|_{}_| <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            else:    
                print('{} -> | {} | {} | <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            
        print('         {}         '.format(player_1_mancala))
        turn = 'P1' if self.current_player == 1 else 'P2'
        print('Turn: ' + turn)
        
    def valid_move(self, pit):
        """
        Function to check if the pit chosen by the current_player is a valid move.
        """
        # Determine the valid range for the current player: if the selected pit is not empty and falls within the allowed pit range
        if self.current_player == 1:
             if pit <= max(self.p1_pits_index) and pit >= min(self.p1_pits_index) and self.board[pit] > 0:
                return True
        
        elif self.current_player == 2:  
            if pit <= max(self.p2_pits_index) and pit >= min(self.p2_pits_index) and self.board[pit] > 0:
                return True
        else:
            return False
        
    def random_move_generator(self):
        """
        Function to generate random valid moves with non-empty pits for the random player
        """
        if self.current_player == 1:
            # Generate a random pit index for Player 1 within their valid range
            pit = random.randint(min(self.p1_pits_index),max(self.p1_pits_index))
            # Continue generating a random pit index until a valid move is found
            while not self.valid_move(pit):
                pit = random.randint(min(self.p1_pits_index),max(self.p1_pits_index))
            return pit + 1

        elif self.current_player == 2:
            pit = random.randint(min(self.p2_pits_index),max(self.p2_pits_index))
            while not self.valid_move(pit):
                pit = random.randint(min(self.p2_pits_index),max(self.p2_pits_index))
            # Adjust the index to match the game's 1-based system by subtracting the offset of pits per player
            return pit - self.pits_per_player
        else: return -1
    
    def play(self, pit):
        """
        This function simulates a single move made by a specific player using their selected pit. It primarily performs three tasks:
        1. It checks if the chosen pit is a valid move for the current player. If not, it prints "INVALID MOVE" and takes no action.
        2. It verifies if the game board has already reached a winning state. If so, it prints "GAME OVER" and takes no further action.
        3. After passing the above two checks, it proceeds to distribute the stones according to the specified Mancala rules.

        Finally, the function then switches the current player, allowing the other player to take their turn.
        """
        print(f"Player {self.current_player} chose pit: {pit}")
        
        # Log the move in a list for later review or display
        self.moves.append((self.current_player, pit))
        
        # Adjust pit index based on player number for easier access to the board list
        if self.current_player == 1: pit = pit - 1
        elif self.current_player == 2: pit = pit + self.pits_per_player
        
        # Check if the game has already been won
        winner, terminal = self.winning_eval()
        if terminal:
            print("GAME OVER")
            return self.board 

        # Validate the move
        if not self.valid_move(pit):
            print("INVALID MOVE")
            return self.board 

        # Begin moving stones from the chosen pit
        temp_node = pit
        n_stones = self.board[temp_node]
        self.board[temp_node] = 0  # Empty the chosen pit

        # Distribute the stones in the subsequent pits
        for i in range(n_stones):
            temp_node = (temp_node + 1) % len(self.board) # Ensure index wraps around the board
            if (self.current_player == 1 and temp_node == self.p2_mancala_index) or \
               (self.current_player == 2 and temp_node == self.p1_mancala_index):
                continue  # Skip adding stones to the opponent's Mancala
            self.board[temp_node] += 1
       
        # Handle capture scenario: If the last stone lands in an empty pit on the player's side
        if self.board[temp_node] == 1:
            if (self.current_player == 1 and self.p1_pits_index[0] <= temp_node <= self.p1_pits_index[1]) or \
               (self.current_player == 2 and self.p2_pits_index[0] <= temp_node <= self.p2_pits_index[1]):
                if self.current_player == 1:
                    opposite_pit = self.p2_pits_index[1] - (temp_node - self.p1_pits_index[0])
                    self.board[self.p1_mancala_index] += self.board[opposite_pit] + 1
                else:
                    opposite_pit = self.p1_pits_index[1] - (temp_node - self.p2_pits_index[0])
                    self.board[self.p2_mancala_index] += self.board[opposite_pit] + 1
                self.board[temp_node] = 0
                self.board[opposite_pit] = 0
        
        # Change the current player
        self.current_player = 2 if self.current_player == 1 else 1
        return self.board
    
    def winning_eval(self):
        """
        Function to verify if the game board has reached the winning state.
        Hint: If either of the players' pits are all empty, then it is considered a winning state.
        """
        # Initialize flags to check if each player's pits are empty
        p1_end = True
        p2_end = True
        
        # Default winner is 0 (no winner yet) and game not ended
        winner = 0
        terminal = False
        
        # Check if all pits for Player 1 are empty
        for i in range(self.p1_pits_index[1]+1):
            if self.board[i] != 0: p1_end = False  # Found a pit with stones, set to False
        
        # Check if all pits for Player 2 are empty
        for i in range(self.p2_pits_index[0],self.p2_pits_index[1]+1):
            if self.board[i] != 0: p2_end = False  # Found a pit with stones, set to False
        if p1_end or p2_end: 
            terminal = True  # If any player's pits are empty, game ends
            if self.board[self.p1_mancala_index] > self.board[self.p2_mancala_index]: winner = 1  # player 1 has more stones
            elif self.board[self.p1_mancala_index] < self.board[self.p2_mancala_index]: winner = 2  # player 2 has more stones
            else: winner = 3
            
        return winner, terminal

### Game Initialization Example:

In [4]:
# Mancala part 1 
game = Mancala(3,2)
game.display_board()

# Player 1 selects pit 1 (1-based index)
game.play(1)
game.display_board()

# Player 2 selects pit 2
game.play(2)
game.display_board()

# Player 1 selects pit 3
game.play(3)
game.display_board()

# Player 2 selects pit 2
game.play(2)
game.display_board()

# Player 1 selects pit 1
game.play(1)
game.display_board()

# Printing the list of moves
print("\nList of valid moves:")
for move in game.moves:
    player, pit = move
    print(f"Player {player} selected pit {pit}")


P1               P2
     ____0____     
1 -> | 2 | 2 | <- 3
2 -> | 2 | 2 | <- 2
3 -> |_2_|_2_| <- 1
         0         
Turn: P1
Player 1 chose pit: 1
P1               P2
     ____0____     
1 -> | 0 | 2 | <- 3
2 -> | 3 | 2 | <- 2
3 -> |_3_|_2_| <- 1
         0         
Turn: P2
Player 2 chose pit: 2
P1               P2
     ____1____     
1 -> | 0 | 3 | <- 3
2 -> | 3 | 0 | <- 2
3 -> |_3_|_2_| <- 1
         0         
Turn: P1
Player 1 chose pit: 3
P1               P2
     ____1____     
1 -> | 0 | 3 | <- 3
2 -> | 3 | 1 | <- 2
3 -> |_0_|_3_| <- 1
         1         
Turn: P2
Player 2 chose pit: 2
P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 3 | 0 | <- 2
3 -> |_0_|_3_| <- 1
         1         
Turn: P1
Player 1 chose pit: 1
INVALID MOVE
P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 3 | 0 | <- 2
3 -> |_0_|_3_| <- 1
         1         
Turn: P1

List of valid moves:
Player 1 selected pit 1
Player 2 selected pit 2
Player 1 selected pit 3
Player 2 

### Build a random player--a player that makes random (legal) move

User vs. Random Player

In [ ]:
game2 = Mancala(6,4)
game2.display_board()
random.seed(int(time.time()))

for i in range(100):
    # Student's move
    pit = int(input("Student, enter your pit choice: "))  # Student chooses a pit
    game2.play(pit)
    game2.display_board()
    winner, terminal = game2.winning_eval()
    if terminal == True:
        if winner == 1:
            print("You Win!")
        elif winner == 2:
            print("Opponent Wins!")
        else:
            print("It's a Draw!")
        break

    # Random player's move
    if game2.current_player == 2:
        random_pit = game2.random_move_generator()  # Generate a valid move
        print(f"Random player chooses pit: {random_pit}")
        game2.play(random_pit)
        game2.display_board()
        winner, terminal = game2.winning_eval()
        if terminal == True:
            if winner == 1:
                print("You Win!")
            elif winner == 2:
                print("Opponent Wins!")
            else:
                print("It's a Draw!")
            break
    else:
        continue

### 100 games of random player against random player:

Output contains each player's respective win rate.

In [5]:
# Context manager to suppress stdout
@contextmanager
def suppress_stdout():
    new_stdout = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = new_stdout
    try:
        yield
    finally:
        sys.stdout = old_stdout

def simulate_games(num_games, pits, stones):
    random.seed(int(time.time()))  # Seed for true randomness
    results = {"Player 1 Wins": 0, "Player 2 Wins": 0, "Draws": 0}
    total_moves_for_wins = 0
    total_wins = 0

    for _ in range(num_games):
        game = Mancala(pits, stones)
        move_count = 0
        while True:
            with suppress_stdout():  # Suppress print statements during the game
                pit = game.random_move_generator()
                game.play(pit)
                move_count += 1
            winner, game_over = game.winning_eval()
            if game_over:
                if winner == 1:
                    results["Player 1 Wins"] += 1
                    total_moves_for_wins += move_count
                    total_wins += 1
                elif winner == 2:
                    results["Player 2 Wins"] += 1
                    total_moves_for_wins += move_count
                    total_wins += 1
                else:
                    results["Draws"] += 1
                break

    if total_wins > 0:
        average_moves_to_win = total_moves_for_wins / total_wins  
    else:
        average_moves_to_win = 0

    results["Average Moves to Win"] = average_moves_to_win

    return results

results = simulate_games(100, 6, 4)
print("Game results after 100 simulations:", results)

Game results after 100 simulations: {'Player 1 Wins': 47, 'Player 2 Wins': 44, 'Draws': 9, 'Average Moves to Win': 38.2967032967033}


# Project Intermediate Results

    So far, I have been able to successfully implement our version of mancala. After a complete run through Homework 7, I have since implemented a random player vs. random player run of the game. This uses the previously implemented random_move_generator function to have two random players play one another. I accomplished this by implementing a simulate_games function that simulates a number of games and tells you the winning rate between players. After running the simulation up to 100,000 times, it seems that the code effectively demonstrates each player winning roughly 50% of the time between runs. I also implemented a context manager to suppress print statements to the console in order to avoid clutter and just recieve the game results as a print statement. In order to do this, I had to import sys, io, and contextmanager libraries. I also imported the time library in order to truly get randomized move inputs compared to running the same random seed over and over. I also buffed out the game's ability to have a user play a random player by running until there is a winner. Beforehand, the game would only run for 5 rounds just to demonstrate the code successfully running. 
    Everything is running smoothly so far and in the coming weeks I plan to construct an AI player for both minimax and alpha beta.

### (A.I) MancalaGame Class Definitions

In [6]:
from games import Game, GameState, random_player, minmax_decision

class MancalaGame(Game):
    def __init__(self, pits_per_player=6, stones_per_pit=4):
        self.pits_per_player = pits_per_player
        self.stones_per_pit = stones_per_pit
        # Create initial board setup
        self.board = [stones_per_pit] * (2 * (pits_per_player + 1))
        self.p1_mancala_index = pits_per_player
        self.p2_mancala_index = len(self.board) - 1
        self.board[self.p1_mancala_index] = 0  # Initialize Mancala for Player 1
        self.board[self.p2_mancala_index] = 0  # Initialize Mancala for Player 2

        # Define ranges for player pits excluding Mancalas
        self.p1_pits_index = list(range(0, pits_per_player))
        self.p2_pits_index = list(range(pits_per_player + 1, self.p2_mancala_index))
        
        # Initialize a list to track moves made during the game
        self.moves = []

        # Initialize GameState
        self.initial = GameState(to_move=1, utility=0, board=self.board, moves=self.p1_pits_index)

    def actions(self, state):
        """ Returns a list of legal moves for the current player """
        moves = []
        start_index = 0 if state.to_move == 1 else self.pits_per_player + 1
        end_index = self.pits_per_player if state.to_move == 1 else len(state.board) - 1

        for i in range(start_index, end_index):
            if state.board[i] > 0:  # Only include non-empty pits
                moves.append(i)
        return moves

    def result(self, state, move):
        """ Execute a move and return the new game state. """
        new_board = list(state.board)
        stones = new_board[move]
        new_board[move] = 0
        index = move
        while stones > 0:
            index = (index + 1) % len(new_board)
            # Skip opponent's Mancala
            if (state.to_move == 2 and index == self.p1_mancala_index) or \
               (state.to_move == 1 and index == self.p2_mancala_index):
                continue
            new_board[index] += 1
            stones -= 1

        # Check if last stone ends in an empty pit on player's side
        if new_board[index] == 1:
            # Check for captures if it's on the player's side
            if (state.to_move == 1 and 0 <= index < self.pits_per_player) or \
               (state.to_move == 2 and self.pits_per_player + 1 <= index < len(new_board) - 1):
                opposite_index = len(new_board) - index - 2
                mancala_index = self.pits_per_player if state.to_move == 1 else len(new_board) - 1
                new_board[mancala_index] += new_board[opposite_index] + 1
                new_board[opposite_index] = 0
                new_board[index] = 0

        # Check if game ends
        p1_empty = all(new_board[i] == 0 for i in range(self.pits_per_player))
        p2_empty = all(new_board[i] == 0 for i in range(self.pits_per_player + 1, len(new_board) - 1))

        new_to_move = 2 if state.to_move == 1 else 1

        return GameState(to_move=new_to_move, utility=0, board=new_board, moves=self.actions(state))
    
    def utility(self, state, player):
        """ Calculate the utility of the state for a player """
        p1_score = state.board[self.pits_per_player]
        p2_score = state.board[-1]
        return (p1_score - p2_score) if player == 1 else (p2_score - p1_score)

    def terminal_test(self, state):
        """ Check if the game is over """
        return all(stone == 0 for stone in state.board[:self.pits_per_player]) or \
               all(stone == 0 for stone in state.board[self.pits_per_player+1:-1])

    def to_move(self, state):
        """ Returns the player whose turn it is """
        return state.to_move

    def display(self, state):
        """ Display the current game state """
        print("Current board:", state.board)
        print("Player 1 Mancala:", state.board[self.pits_per_player])
        print("Player 2 Mancala:", state.board[-1])
        print("Player to move:", "Player 1" if state.to_move == 1 else "Player 2")
        
    def minimax_decision(self, state, depth):
        """Determine the best action by exploring depth-limited Minimax tree."""
        player = state.to_move

        def max_value(state, depth):
            if self.terminal_test(state) or depth == 0:
                return self.utility(state, player)
            v = float('-inf')
            for a in self.actions(state):
                v = max(v, min_value(self.result(state, a), depth - 1))
            return v

        def min_value(state, depth):
            if self.terminal_test(state) or depth == 0:
                return self.utility(state, player)
            v = float('inf')
            for a in self.actions(state):
                v = min(v, max_value(self.result(state, a), depth - 1))
            return v

        # Use max_value to get the best possible move for the current player
        best_score = float('-inf')
        best_action = None
        for action in self.actions(state):
            value = min_value(self.result(state, action), depth - 1)
            if value > best_score:
                best_score = value
                best_action = action
        return best_action
    
    def play_game(self, depth):
        state = self.initial
        while not self.terminal_test(state):
            if state.to_move == 1:
                # Minimax player
                move = self.minimax_decision(state, depth)
            else:
                # Random player
                move = random_player(self, state)
            self.moves.append(move)  # Append move to the list
            state = self.result(state, move)
        return self.utility(state, 1)  # Return the utility for Player 1
    
    def alpha_beta_decision(self, state, depth):
        """Determine the best action by exploring the game tree using Alpha-Beta pruning."""
        player = self.to_move(state)

        def max_value(state, alpha, beta, depth):
            if self.terminal_test(state) or depth == 0:
                return self.utility(state, player)
            v = float('-inf')
            for a in self.actions(state):
                v = max(v, min_value(self.result(state, a), alpha, beta, depth-1))
                if v >= beta:
                    return v
                alpha = max(alpha, v)
            return v

        def min_value(state, alpha, beta, depth):
            if self.terminal_test(state) or depth == 0:
                return self.utility(state, player)
            v = float('inf')
            for a in self.actions(state):
                v = min(v, max_value(self.result(state, a), alpha, beta, depth-1))
                if v <= alpha:
                    return v
                beta = min(beta, v)
            return v

        best_score = float('-inf')
        best_action = None
        for action in self.actions(state):
            value = min_value(self.result(state, action), best_score, float('inf'), depth-1)
            if value > best_score:
                best_score = value
                best_action = action
        return best_action

## A.I. Player -> Minimax

### Minimax Usage Example

In [7]:
def play_game_with_minimax(game, depth):
    start_time = time.time()
    
    state = game.initial
    while not game.terminal_test(state):
        game.display(state)
        move = game.minimax_decision(state, depth)
        print(f"Player {state.to_move} chooses pit: {move}")
        state = game.result(state, move)
    game.display(state)
    final_utility_p1 = game.utility(state, 1)
    final_utility_p2 = game.utility(state, 2)
    if final_utility_p1 > 0:
        winner = "Player 1"
    elif final_utility_p2 > 0:
        winner = "Player 2"
    else:
        winner = "Tie"
    end_time = time.time()

    print(f"Winner: {winner}")
    print(f"Game duration: {end_time - start_time:.2f} seconds")

mancala_game = MancalaGame()
play_game_with_minimax(mancala_game, depth=5)

Current board: [4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0]
Player 1 Mancala: 0
Player 2 Mancala: 0
Player to move: Player 1
Player 1 chooses pit: 5
Current board: [4, 4, 4, 4, 4, 0, 1, 5, 5, 5, 4, 4, 4, 0]
Player 1 Mancala: 1
Player 2 Mancala: 0
Player to move: Player 2
Player 2 chooses pit: 7
Current board: [4, 4, 4, 4, 4, 0, 1, 0, 6, 6, 5, 5, 5, 0]
Player 1 Mancala: 1
Player 2 Mancala: 0
Player to move: Player 1
Player 1 chooses pit: 1
Current board: [4, 0, 5, 5, 5, 0, 2, 0, 6, 6, 5, 5, 5, 0]
Player 1 Mancala: 2
Player 2 Mancala: 0
Player to move: Player 2
Player 2 chooses pit: 8
Current board: [5, 0, 5, 5, 5, 0, 2, 0, 0, 7, 6, 6, 6, 1]
Player 1 Mancala: 2
Player 2 Mancala: 1
Player to move: Player 1
Player 1 chooses pit: 0
Current board: [0, 1, 6, 6, 6, 0, 3, 0, 0, 7, 6, 6, 6, 1]
Player 1 Mancala: 3
Player 2 Mancala: 1
Player to move: Player 2
Player 2 chooses pit: 9
Current board: [1, 2, 7, 6, 6, 0, 3, 0, 0, 0, 7, 7, 7, 2]
Player 1 Mancala: 3
Player 2 Mancala: 2
Player to move: Play

### AI (Minimax) vs Random

In [8]:
def simulate_games(num_games, depth):
    results = {
        "Player 1 Wins": 0, 
        "Player 2 Wins": 0, 
        "Draws": 0, 
        "Average Moves to Win": 0,
        "Average Game Duration": 0
    }
    total_moves = 0
    winning_games = 0
    total_duration = 0

    for _ in range(num_games):
        start_time = time.time()

        game = MancalaGame(pits_per_player=6, stones_per_pit=4)
        final_utility = game.play_game(depth)
        game_duration = time.time() - start_time

        total_duration += game_duration

        if final_utility > 0:
            results["Player 1 Wins"] += 1
            total_moves += len(game.moves)
            winning_games += 1
        elif final_utility < 0:
            results["Player 2 Wins"] += 1
            total_moves += len(game.moves)
            winning_games += 1
        else:
            results["Draws"] += 1

    if winning_games > 0:
        results["Average Moves to Win"] = total_moves / winning_games
    results["Average Game Duration"] = total_duration / num_games

    return results

# Run the simulation for 100 games with a depth of 5 plies
simulation_results = simulate_games(100, 5)
print("Simulation Results:", simulation_results)

Simulation Results: {'Player 1 Wins': 100, 'Player 2 Wins': 0, 'Draws': 0, 'Average Moves to Win': 27.77, 'Average Game Duration': 0.1573225712776184}


### Overwhelming Success Verification

In [9]:
def play_game_with_logging(game, depth):
    state = game.initial
    move_number = 0
    while not game.terminal_test(state):
        print(f"Move {move_number}: Player {state.to_move}'s turn.")
        move = game.minimax_decision(state, depth) if state.to_move == 1 else random_player(game, state)
        print(f"Player {state.to_move} chooses pit: {move}")
        state = game.result(state, move)
        print(f"Board after move: {state.board}")
        print(f"Current utility: Player 1: {game.utility(state, 1)}, Player 2: {game.utility(state, 2)}\n")
        move_number += 1
    print("Final board:", state.board)
    print("Game over. Final utility:", game.utility(state, 1))
    return state

detailed_game = MancalaGame()
play_game_with_logging(detailed_game, 5)

Move 0: Player 1's turn.
Player 1 chooses pit: 5
Board after move: [4, 4, 4, 4, 4, 0, 1, 5, 5, 5, 4, 4, 4, 0]
Current utility: Player 1: 1, Player 2: -1

Move 1: Player 2's turn.
Player 2 chooses pit: 12
Board after move: [5, 5, 5, 4, 4, 0, 1, 5, 5, 5, 4, 4, 0, 1]
Current utility: Player 1: 0, Player 2: 0

Move 2: Player 1's turn.
Player 1 chooses pit: 0
Board after move: [0, 6, 6, 5, 5, 0, 7, 0, 5, 5, 4, 4, 0, 1]
Current utility: Player 1: 6, Player 2: -6

Move 3: Player 2's turn.
Player 2 chooses pit: 8
Board after move: [0, 6, 6, 5, 5, 0, 7, 0, 0, 6, 5, 5, 1, 2]
Current utility: Player 1: 5, Player 2: -5

Move 4: Player 1's turn.
Player 1 chooses pit: 2
Board after move: [0, 6, 0, 6, 6, 1, 8, 1, 1, 6, 5, 5, 1, 2]
Current utility: Player 1: 6, Player 2: -6

Move 5: Player 2's turn.
Player 2 chooses pit: 12
Board after move: [0, 6, 0, 6, 6, 1, 8, 1, 1, 6, 5, 5, 0, 3]
Current utility: Player 1: 5, Player 2: -5

Move 6: Player 1's turn.
Player 1 chooses pit: 1
Board after move: [0, 0, 1

GameState(to_move=1, utility=0, board=[0, 0, 1, 0, 0, 2, 31, 0, 0, 0, 0, 0, 0, 14], moves=[10])

### Is this AI player that uses Minimax better than random chance?

The simulation results, where the AI player employing minimax competed against a randomly moving opponent, showed that the AI player won 100% of the games. This overwhelming success rate indicates that the minimax AI is significantly better than random chance. I didn't believe the success rate at first but then verified the effectiveness of the algorithm by analyzing the details of an individual game between the AI and random player. 

The minimax algorithm gives the AI player a strategic advantage by evaluating the possible outcomes of moves several plies ahead, based on the utility function, which in this game is the difference in the number of stones in each player's Mancala. Unlike the random player, which makes moves based on no predictive evaluation, the minimax AI chooses the move that maximizes its chances of ending up with a favorable game state. This forward-thinking strategy allows the AI to not only react to the current game state but also to plan for future moves, considering the opponent's possible responses. Moreover, the structure of Mancala as a game where future game states are relatively predictable based on current decisions makes it particularly suitable for the minimax algorithm. The ability to anticipate and strategically block the opponent's potential gains or exploit their weaknesses by calculating several moves ahead contributes to the AI's high win rate. This predictive power, coupled with a solid utility function that clearly aligns with winning conditions, confirms that the AI player is indeed far superior to a strategy based purely on randomness.

## A.I. Player -> Alpha Beta

In [10]:
def play_game_with_alpha_beta(game, depth):
    start_time = time.time()
    
    state = game.initial
    move_number = 0
    while not game.terminal_test(state):
        print(f"Move {move_number}: Player {state.to_move}'s turn.")
        if state.to_move == 1:
            move = game.alpha_beta_decision(state, depth)
        else:
            move = random_player(game, state)
        print(f"Player {state.to_move} chooses pit: {move}")
        state = game.result(state, move)
        print(f"Board after move: {state.board}")
        # To reflect utility for both players properly
        print(f"Current utility: Player 1: {game.utility(state, 1)}, Player 2: {game.utility(state, 2)}\n")
        move_number += 1
    
    end_time = time.time()
    print("Final board:", state.board)
    # Print final utility for both players
    print(f"Game over. Final utility: Player 1: {game.utility(state, 1)}, Player 2: {game.utility(state, 2)}")
    print(f"Game duration: {end_time - start_time:.2f} seconds")
    return state

alpha_beta_game = MancalaGame()
play_game_with_alpha_beta(alpha_beta_game, depth=5)

Move 0: Player 1's turn.
Player 1 chooses pit: 5
Board after move: [4, 4, 4, 4, 4, 0, 1, 5, 5, 5, 4, 4, 4, 0]
Current utility: Player 1: 1, Player 2: -1

Move 1: Player 2's turn.
Player 2 chooses pit: 12
Board after move: [5, 5, 5, 4, 4, 0, 1, 5, 5, 5, 4, 4, 0, 1]
Current utility: Player 1: 0, Player 2: 0

Move 2: Player 1's turn.
Player 1 chooses pit: 0
Board after move: [0, 6, 6, 5, 5, 0, 7, 0, 5, 5, 4, 4, 0, 1]
Current utility: Player 1: 6, Player 2: -6

Move 3: Player 2's turn.
Player 2 chooses pit: 9
Board after move: [1, 6, 6, 5, 5, 0, 7, 0, 5, 0, 5, 5, 1, 2]
Current utility: Player 1: 5, Player 2: -5

Move 4: Player 1's turn.
Player 1 chooses pit: 1
Board after move: [1, 0, 7, 6, 6, 1, 8, 1, 5, 0, 5, 5, 1, 2]
Current utility: Player 1: 6, Player 2: -6

Move 5: Player 2's turn.
Player 2 chooses pit: 11
Board after move: [2, 1, 8, 6, 6, 1, 8, 1, 5, 0, 5, 0, 2, 3]
Current utility: Player 1: 5, Player 2: -5

Move 6: Player 1's turn.
Player 1 chooses pit: 2
Board after move: [2, 1, 0

GameState(to_move=2, utility=0, board=[0, 0, 0, 0, 0, 0, 21, 1, 11, 3, 1, 1, 2, 8], moves=[5])

### AI (Alpha Beta) vs Random

In [11]:
def simulate_alpha_beta_games(num_games, depth):
    results = {
        "Player 1 Wins": 0, 
        "Player 2 Wins": 0, 
        "Draws": 0, 
        "Average Moves to Win": 0,
        "Average Game Duration": 0
    }
    total_moves_for_wins = 0
    total_win_games = 0
    total_duration = 0

    for _ in range(num_games):
        start_time = time.time()
        game = MancalaGame()
        state = game.initial
        move_count = 0

        while not game.terminal_test(state):
            if state.to_move == 1:
                move = game.alpha_beta_decision(state, depth)
            else:
                move = random_player(game, state)
            state = game.result(state, move)
            move_count += 1

        game_duration = time.time() - start_time
        total_duration += game_duration

        final_utility = game.utility(state, 1)
        
        if final_utility > 0:
            results["Player 1 Wins"] += 1
            total_moves_for_wins += move_count
            total_win_games += 1
        elif final_utility < 0:
            results["Player 2 Wins"] += 1
            total_moves_for_wins += move_count
            total_win_games += 1
        else:
            results["Draws"] += 1

    if total_win_games > 0:
        results["Average Moves to Win"] = total_moves_for_wins / total_win_games
    results["Average Game Duration"] = total_duration / num_games 

    return results

# Run the simulation for 100 games with a depth of 5 plies
alpha_beta_simulation_results = simulate_alpha_beta_games(100, 5)
print("Simulation Results:", alpha_beta_simulation_results)

Simulation Results: {'Player 1 Wins': 100, 'Player 2 Wins': 0, 'Draws': 0, 'Average Moves to Win': 28.81, 'Average Game Duration': 0.04418285846710205}


### Are the results from our Alpha Beta player different from those from our Minimax player?
In our simulations comparing Alpha Beta pruning with the Minimax algorithm for Mancala, both methods achieved a 100% success rate against a random opponent. This demonstrates that both strategies are equally effective in winning games by accurately predicting the best moves. However, the main difference lies in their efficiency: Alpha Beta pruning enhances performance by eliminating unnecessary branches in the game tree, leading to quicker decision-making without compromising the outcome. For reference, the average game duration for Minimax was 0.17 seconds and only 0.04 seconds for Alpha Beta.

Despite the speed advantage of Alpha Beta, the consistent winning outcomes across both algorithms suggest that the complexity of Mancala, especially against a non-strategic opponent, doesn't fully challenge the capabilities of either algorithm. The game's simplicity allows both Alpha Beta and Minimax to achieve perfect results, with Alpha Beta's primary advantage being faster computational times, not necessarily better strategic performance.

### Extra Credit: Increased Depth Alpha Beta

In [ ]:
alpha_beta_simulation_results = simulate_alpha_beta_games(100, 10)
print("Simulation Results:", alpha_beta_simulation_results)

#### Does increasing the number of plies improve the play for our AI player? Why or why not?

In this case because we already are so successful with just 5 piles, 10 cannot make it any better. However, expanding the number of games significantly could show a difference in results. Interestingly enough, this increased depth with the same number of games brought the average game duration from 0.04 seconds to 5.54 seconds. This indicates the significant computational cost associated with exploring deeper levels in the decision tree. This dramatic increase in duration suggests that the additional depth explores many more possible game states, requiring more processing power and time. I am interested in later examinations to dive deeper into this idea using a larger number of games to see how the true effectiveness of the algorithm changes with depth. 

## Project Report
In this final project, we examined the efficacy of AI strategies, specifically Minimax and Alpha Beta pruning algorithms, in the game of Mancala. Both algorithms demonstrated a 100% success rate against a random opponent across multiple simulations, highlighting their effectiveness in strategic decision-making. While the results were overwhelmingly positive, showcasing the power of both algorithms in a controlled game environment, the primary distinction lay in the computational efficiency; Alpha Beta pruning consistently outperformed Minimax in speed without sacrificing accuracy or effectiveness.

Further investigation revealed that although both AI methods secured perfect scores, the simplicity of Mancala as a game means that there's minimal challenge to their strategic capacities, particularly against non-strategic, random moves. Alpha Beta's main advantage is its ability to reduce computational overhead by pruning unnecessary branches in the decision tree, thereby accelerating the decision-making process. This project not only reinforced the algorithms' potential in game strategy but also underlined the importance of choosing the right algorithm based on the operational needs and complexity of the task.